In [ ]:
import pandas as pd
import re 
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

'Simple script for sorting movies alphabetically'

# Reading CSV files into dataframes
movies_df = pd.read_csv("movies.csv", low_memory = False)
movies_df['clean_title'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()

# Cleaning up movie titles that start with "The", "A", or "An" and moving them to front of title
movies_df['title'] = movies_df['title'].str.replace(
    r'^(.*),\s*(The|A|An)(\s+\(\d{4}\))?$', 
    r'\2 \1\3', 
    regex=True, 
    case=False
).str.strip()

# Redo clean_title column after adjusting title format 
movies_df['clean_title'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()

# Reading rating CSV file, skip bad lines since an error was returned 
ratings_df = pd.read_csv("ratings.csv", dtype={'userId': int, 'movieId': int, 'rating': float}, on_bad_lines='skip')

# Sorting the movies alphabetically, ignore special characters / case
movies_df['sort_key'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()
movies_df = movies_df.sort_values(by = 'sort_key', ascending = True)
movies_df = movies_df.drop(columns=['sort_key']).reset_index(drop=True)

# Cleaning genre column by removing pipe separator
movies_df['genres_clean'] = movies_df['genres'].str.replace('|', ' ', regex = False)

# New column that combines title and cleaned genres
movies_df['features'] = movies_df['title'] + " " + movies_df['genres_clean']

# Creating a movies df with simpler format, to allow for compatibility with user input

tfidf = TfidfVectorizer(stop_words='english')

# Constructing TF-IDF matrix (rows = movies, columns = unique vocabulary words)
tfidf_matrix = tfidf.fit_transform(movies_df['features'])

# Use cosine similarity to find similarity between all movies based off of TF-IDF
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


# Create a series mapping movie titles to index in dataframe for easier lookup
indices = pd.Series(movies_df.index, index=movies_df['clean_title']).drop_duplicates()


'''Here we create the main function, that takes a movie and returns a list of the top 10 most similar movies based off the cos similarity matrix established'''
def get_recommendations(title, cosine_sim = cosine_sim):

    title_clean = re.sub(r'[^a-zA-Z0-9\s]', '', title).lower().strip()
    # First check if there is an exact match for the title in the dataset
    if title_clean not in indices:
    # Check if we can find a partial match for the title in the dataset
        matching_rows = movies_df[movies_df['clean_title'].str.contains(title_clean, regex=False)]
        
        if not matching_rows.empty:
            # Look for the partial with the shortest title length as it is likely to be most relevant
            matching_rows = matching_rows.copy()
            matching_rows['title_len'] = matching_rows['clean_title'].str.len()
            matching_rows = matching_rows.sort_values(by='title_len', ascending=True)

            title_clean = matching_rows['clean_title'].iloc[0]
            matched_original_title = matching_rows['title'].iloc[0]
            print(f"(Matched '{title}' to '{matched_original_title}')")
        else:
            return f"Movie {title} not found in the dataset. Please check for typos or try a different movie."
    # get index of movie that matches title
    idx = indices[title_clean]

    # get pairwise similarity scores of all movies with given movie 
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # sort from most simliar to least and choose top 10
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)


    # We are not interested in sequel recommendations 
    # Here consider a base title that ignores numbers that indicate parts 
    base_title_clean = re.sub(r'\b(2|3|4|5|ii|iii|iv|v|part\s*\d+|vol\s*\d+)\b', '', title_clean).strip()
    base_title_clean = re.sub(r'\s*\d{4}\s*', '', base_title_clean).strip()

    filtered_scores = []
    for movie_idx, score in similarity_scores:
        # Skip the movie itself (score will be 1.0 or highest index match)
        if movie_idx == idx:
            continue
            
        candidate_title = movies_df['clean_title'].iloc[movie_idx]
        
        # Check if canditate title has the same base title as the input (e.g Toy Story 1, Toy Story 2)
        if base_title_clean and base_title_clean in candidate_title:
            # Skip this candidate because it's a sequel/prequel/spin-off in the same series
            continue
            
        filtered_scores.append((movie_idx, score))
        
        # Stop once we have collected the top 10 non-sequel recommendations
        if len(filtered_scores) == 10:
            break
    movie_indices = [i[0] for i in filtered_scores]
    return movies_df['title'].iloc[movie_indices].tolist()

# Input system for the user to enter a movie title and get recommendations
print("Movie recommendation system. Type quit or exit to leave the program at any time. \nPlease enter a movie title to get recommendations :")
while True:
    user_input = input("Movie Title :")
    if user_input.lower() in ['quit', 'exit']:
        print("Ending program.")
        break
    recommendations = get_recommendations(user_input)

    # Use isinstance to prevent errors if input is not in dataset
    if isinstance(user_input, str) and isinstance(recommendations,list):
        for i, movie in enumerate(recommendations, 1):
            print(f"  {i}. {movie}")
        print("-" * 40)
    else:
        print(recommendations)
        print("-" * 40)

Movie recommendation system. Type quit or exit to leave the program at any time. 
Please enter a movie title to get recommendations :
(Matched 'the dark knight' to 'The Dark Knight (2008)')
  1. First Knight (1995)
  2. Alone in the Dark II (2008)
  3. Batman: Gotham Knight (2008)
  4. Knight and Day (2010)
  5. A Knight's Tale (2001)
  6. Dark Blue (2003)
  7. A Cry in the Dark (1988)
  8. Never Back Down (2008)
  9. Dark Shadows (2012)
  10. Black Knight (2001)
----------------------------------------
(Matched 'quir' to 'Squirm (1976)')
  1. The Omen (1976)
  2. Carrie (1976)
  3. Rocky (1976)
  4. Network (1976)
  5. The Shaggy D.A. (1976)
  6. Burnt Offerings (1976)
  7. Silent Movie (1976)
  8. The Enforcer (1976)
  9. King Kong (1976)
  10. Obsession (1976)
----------------------------------------
Ending program.
